In [1]:
# ----------------------------------------------------------------------------
# Title: Term Project - Milestone 3
# Author: Surenther Selvaraj
# Date: 24 Apr 2026
# Modified By: Surenther Selvaraj
# ----------------------------------------------------------------------------

### Child Care Affordability Dashboard 

In [1]:
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

# 1. Load and Prepare Data
df = pd.read_excel('nationaldatabaseofchildcareprices.xlsx')
df['Infant_Ratio'] = (df['MCInfant'] * 52) / df['MHI']
df['Affordability_Gap'] = df['Infant_Ratio'] - 0.07

app = dash.Dash(__name__)

# 2. Dashboard Layout
app.layout = html.Div(style={'backgroundColor': '#f9f9f9', 'padding': '20px', 'fontFamily': 'Arial'}, children=[
    html.H1("Childcare Affordability Crisis Dashboard (2008-2018)", style={'textAlign': 'center', 'color': '#2c3e50'}),
    
    # Year Slider
    html.Div([
        html.Label("Select Year to Animate Changes:", style={'fontWeight': 'bold'}),
        dcc.Slider(
            id='year-slider',
            min=df['StudyYear'].min(),
            max=df['StudyYear'].max(),
            value=df['StudyYear'].min(),
            marks={str(year): str(year) for year in df['StudyYear'].unique()},
            step=None
        )
    ], style={'marginBottom': '40px', 'padding': '0 50px'}),

    html.Div(className='row', style={'display': 'flex'}, children=[
        # Choropleth Map
        html.Div([
            dcc.Graph(id='choropleth-map')
        ], style={'width': '65%'}),

        #  Leaderboard
        html.Div([
            html.H3("Top 10 High-Burden Counties", style={'fontSize': '18px'}),
            dash_table.DataTable(
                id='leaderboard-table',
                columns=[{"name": "County", "id": "County_Name"}, {"name": "Burden %", "id": "Infant_Ratio"}],
                style_cell={'textAlign': 'left', 'padding': '5px'},
                style_header={'backgroundColor': '#b2182b', 'color': 'white', 'fontWeight': 'bold'}
            )
        ], style={'width': '35%', 'paddingLeft': '20px'})
    ]),

    # Provider Comparison
    html.Div([
        dcc.Graph(id='provider-bar')
    ], style={'marginTop': '40px'})
])

# 3. Interactivity 
@app.callback(
    [Output('choropleth-map', 'figure'),
     Output('provider-bar', 'figure'),
     Output('leaderboard-table', 'data')],
    [Input('year-slider', 'value')]
)
def update_dashboard(selected_year):
    year_df = df[df['StudyYear'] == selected_year]
    
    # Map update
    fig_map = px.choropleth(
        year_df, locations="State_Abbreviation", locationmode="USA-states",
        color="Affordability_Gap", scope="usa", color_continuous_scale="OrRd",
        range_color=[0, 0.2], title=f"The Affordability Gap in {selected_year}"
    )

    # Bar Chart update
    bar_df = year_df.groupby('State_Abbreviation')[['MCInfant', 'MFCCInfant']].mean().reset_index()
    fig_bar = px.bar(
        bar_df, x='State_Abbreviation', y=['MCInfant', 'MFCCInfant'],
        barmode='group', title="Center vs. Family-Based Weekly Costs",
        color_discrete_sequence=['#b2182b', '#7f7f7f']
    )

    # Leaderboard update
    lb_data = year_df.sort_values(by='Infant_Ratio', ascending=False).head(10)
    lb_data['Infant_Ratio'] = (lb_data['Infant_Ratio'] * 100).round(2).astype(str) + '%'
    
    return fig_map, fig_bar, lb_data.to_dict('records')

if __name__ == '__main__':
    app.run(debug=True)

### Design Decisions Narrative: Childcare Affordability Dashboard

#### Visual Strategy & Hierarchy
The design of this dashboard is governed by a "Red Alert" hierarchy, specifically tailored for a legislative audience. Because the core finding reveals a national burden of 16.39%—more than double the federal benchmark—the Visual A Choropleth Map serves as the "Hero" element. It occupies the largest portion of the screen to leverage spatial processing, allowing legislators to immediately identify regional "crisis zones" without having to parse raw tables.

#### Color Theory
I utilized a sequential "OrRd" (Orange-Red) color palette. In data visualization for public policy, this palette effectively communicates risk and urgency. Neutral whites and light oranges represent areas nearing the 7% HHS threshold, while deep crimson signifies counties reaching the 54.3% maximum burden. This psychological color association ensures that the "Affordability Gap" is felt intuitively by the viewer before the specific numbers are even read.

#### Sizing, Spacing, and Typography
To maintain a professional, "Government Report" aesthetic, I prioritized generous white space around the leaderboard and bar charts. This prevents cognitive overload and ensures that the Visual D Year Scrubber remains the focal point for interaction. High-density charts, such as the provider comparison, use high-contrast Slate Grey and Crimson to differentiate between Center-based and Family-based care, making the cost-benefit analysis unmistakable even on a projected screen. The typography is a clean, sans-serif face, optimized for legibility during screen recording and video playback.